# 01_dlt_gold_pipeline

## Gold Layer Data Model (DLT)

### Dimensions
- `dim_date` - Date hierarchy (year, quarter, month)
- `dim_region` - Region hierarchy (state, metro, county, city, zip)
- `dim_property_type` - Property type categories

### Facts
- `fact_housing_metrics` - Core housing metrics by region/date

### Aggregations
- `agg_monthly_regional` - Monthly stats by state/metro
- `agg_top_regions_value` - Top 10 regions by home value
- `agg_top_regions_growth` - Top 10 regions by YoY growth
- `agg_yearly_trend` - Annual trend summary

## Note
This notebook runs as a DLT pipeline. Deploy via `resources/pipelines.yml`.

In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# ==================== DIMENSION: DATE ====================
@dlt.table(
    name="dim_date",
    comment="Date dimension with year, quarter, month hierarchy"
)
def dim_date():
    # Get distinct dates from Silver time series
    df = spark.table("zillow.zillow_silver_dlt.zip_ts_silver").select("date").distinct()
    
    return (df
        .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
        .withColumn("year", F.year("date"))
        .withColumn("quarter", F.quarter("date"))
        .withColumn("month", F.month("date"))
        .withColumn("month_name", F.date_format("date", "MMMM"))
        .withColumn("day_of_month", F.dayofmonth("date"))
        .withColumn("day_of_week", F.dayofweek("date"))
        .withColumn("week_of_year", F.weekofyear("date"))
        .withColumn("year_month", F.date_format("date", "yyyy-MM"))
        .withColumn("year_quarter", F.concat(F.col("year"), F.lit("-Q"), F.col("quarter")))
        .withColumn("is_month_start", F.col("day_of_month") == 1)
        .withColumn("is_month_end", F.col("date") == F.last_day("date"))
        .select(
            "date_key",
            "date",
            "year",
            "quarter",
            "month",
            "month_name",
            "year_month",
            "year_quarter",
            "day_of_month",
            "day_of_week",
            "week_of_year",
            "is_month_start",
            "is_month_end"
        )
    )

In [0]:
# ==================== DIMENSION: REGION ====================
@dlt.table(
    name="dim_region",
    comment="Region dimension with state, metro, county, city hierarchy"
)
def dim_region():
    # Use the Silver dim_region if exists, otherwise build from crosswalks
    try:
        df = spark.table("zillow.zillow_silver_dlt.dim_region")
    except:
        # Fallback: build from crosswalks
        cities = spark.table("zillow.zillow_silver_dlt.cities_crosswalk_silver")
        counties = spark.table("zillow.zillow_silver_dlt.county_crosswalk_silver")
        
        df = (cities
            .join(counties,
                  (F.lower(cities.county) == F.lower(counties.county_name)) &
                  (F.lower(cities.state) == F.lower(counties.state_name)),
                  "left")
            .select(
                F.md5(F.concat_ws("|", cities.unique_city_id, cities.city, cities.state)).alias("region_key"),
                F.lit("city").alias("region_level"),
                cities.unique_city_id,
                cities.city,
                cities.county,
                cities.state,
                counties.fips,
                counties.metro_name_zillow.alias("metro"),
                counties.cbsa_name,
                counties.cbsa_code
            )
        )
    
    return (df
        .withColumn("region_key", F.coalesce(F.col("region_key"), F.md5(F.col("city"))))
        .dropDuplicates(["region_key"])
    )

In [0]:
# ==================== DIMENSION: PROPERTY TYPE ====================
@dlt.table(
    name="dim_property_type",
    comment="Property type dimension with categories and descriptions"
)
def dim_property_type():
    # Static dimension for property types in Zillow data
    data = [
        (1, "all_homes", "All Homes", "Single-family, condo, and co-op homes"),
        (2, "single_family_residence", "Single Family", "Single-family detached homes"),
        (3, "condo_coop", "Condo/Co-op", "Condominiums and co-operatives"),
        (4, "1_bedroom", "1 Bedroom", "Properties with 1 bedroom"),
        (5, "2_bedroom", "2 Bedroom", "Properties with 2 bedrooms"),
        (6, "3_bedroom", "3 Bedroom", "Properties with 3 bedrooms"),
        (7, "4_bedroom", "4 Bedroom", "Properties with 4 bedrooms"),
        (8, "5_bedroom_or_more", "5+ Bedroom", "Properties with 5 or more bedrooms"),
        (9, "bottom_tier", "Bottom Tier", "Lower third of home values in metro"),
        (10, "middle_tier", "Middle Tier", "Middle third of home values in metro"),
        (11, "top_tier", "Top Tier", "Top third of home values in metro"),
        (12, "duplex_triplex", "Duplex/Triplex", "2-3 unit buildings"),
        (13, "multi_family_5_plus", "Multi-Family 5+", "5+ unit buildings"),
    ]
    
    return spark.createDataFrame(data, 
        ["property_type_key", "property_type_code", "property_type_name", "description"]
    )

In [0]:
# ==================== FACT: HOUSING METRICS ====================
@dlt.table(
    name="fact_housing_metrics",
    comment="Fact table: Core housing metrics by region and date",
    partition_cols=["year", "month"]
)
@dlt.expect_or_drop("valid_date", "date IS NOT NULL")
@dlt.expect_or_drop("valid_region", "region_name IS NOT NULL")
def fact_housing_metrics():
    # Combine all geographic levels into unified fact table
    tables = [
        ("zillow.zillow_silver_dlt.city_ts_silver", "city"),
        ("zillow.zillow_silver_dlt.county_ts_silver", "county"),
        ("zillow.zillow_silver_dlt.metro_ts_silver", "metro"),
        ("zillow.zillow_silver_dlt.zip_ts_silver", "zip"),
    ]
    
    dfs = []
    
    # Core metrics to extract (common across all levels)
    core_cols = [
        "date", "region_name",
        "zhvi_all_homes", "zhvi_single_family_residence", "zhvi_condo_coop",
        "zhvi_bottom_tier", "zhvi_middle_tier", "zhvi_top_tier",
        "zri_all_homes",
        "median_listing_price_all_homes",
        "median_listing_price_per_sqft_all_homes",
        "inventory_raw_all_homes",
        "price_to_rent_ratio_all_homes",
        "pct_of_homes_increasing_in_values_all_homes",
        "pct_of_homes_decreasing_in_values_all_homes"
    ]
    
    for table_name, level in tables:
        try:
            df = spark.table(table_name)
            # Select only columns that exist
            available_cols = [c for c in core_cols if c in df.columns]
            df = (df
                .select(*available_cols)
                .withColumn("region_level", F.lit(level))
            )
            dfs.append(df)
        except:
            pass
    
    if not dfs:
        # Return empty DataFrame with schema if no data
        return spark.createDataFrame([], "date date, region_name string, region_level string")
    
    # Union all levels
    from functools import reduce
    combined = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)
    
    return (combined
        .withColumn("year", F.year("date"))
        .withColumn("month", F.month("date"))
        .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
        .withColumn("region_key", F.md5(F.concat_ws("|", "region_name", "region_level")))
        .withColumn("fact_key", F.md5(F.concat_ws("|", "date", "region_name", "region_level")))
    )

In [0]:
# ==================== AGGREGATION: MONTHLY REGIONAL ====================
@dlt.table(
    name="agg_monthly_regional",
    comment="Monthly aggregated housing metrics by region level and state"
)
def agg_monthly_regional():
    fact = dlt.read("fact_housing_metrics")
    
    return (fact
        .groupBy("year", "month", "region_level")
        .agg(
            F.count("*").alias("record_count"),
            F.countDistinct("region_name").alias("region_count"),
            F.avg("zhvi_all_homes").alias("avg_zhvi"),
            F.min("zhvi_all_homes").alias("min_zhvi"),
            F.max("zhvi_all_homes").alias("max_zhvi"),
            F.percentile_approx("zhvi_all_homes", 0.5).alias("median_zhvi"),
            F.avg("zri_all_homes").alias("avg_zri"),
            F.avg("median_listing_price_all_homes").alias("avg_listing_price"),
            F.sum("inventory_raw_all_homes").alias("total_inventory"),
            F.avg("price_to_rent_ratio_all_homes").alias("avg_price_to_rent")
        )
        .withColumn("year_month", F.concat(F.col("year"), F.lit("-"), F.lpad(F.col("month"), 2, "0")))
        .orderBy("year", "month", "region_level")
    )

In [0]:
# ==================== AGGREGATION: TOP 10 BY VALUE ====================
@dlt.table(
    name="agg_top_regions_by_value",
    comment="Top 10 regions by average home value (ZHVI) - latest month"
)
def agg_top_regions_by_value():
    fact = dlt.read("fact_housing_metrics")
    
    # Get latest date
    latest_date = fact.agg(F.max("date")).collect()[0][0]
    
    return (fact
        .filter(F.col("date") == latest_date)
        .filter(F.col("region_level") == "city")  # Focus on cities
        .filter(F.col("zhvi_all_homes").isNotNull())
        .select(
            "region_name",
            "region_level",
            "zhvi_all_homes",
            "zri_all_homes",
            "median_listing_price_all_homes",
            "price_to_rent_ratio_all_homes",
            "date"
        )
        .orderBy(F.desc("zhvi_all_homes"))
        .limit(10)
        .withColumn("rank", F.row_number().over(Window.orderBy(F.desc("zhvi_all_homes"))))
    )

In [0]:
# ==================== AGGREGATION: TOP 10 BY GROWTH ====================
@dlt.table(
    name="agg_top_regions_by_growth",
    comment="Top 10 regions by YoY home value growth"
)
def agg_top_regions_by_growth():
    fact = dlt.read("fact_housing_metrics")
    
    # Calculate YoY growth
    window = Window.partitionBy("region_name", "region_level").orderBy("date")
    
    with_growth = (fact
        .filter(F.col("zhvi_all_homes").isNotNull())
        .withColumn("zhvi_1y_ago", F.lag("zhvi_all_homes", 12).over(window))
        .filter(F.col("zhvi_1y_ago").isNotNull())
        .withColumn("yoy_growth_pct", 
            ((F.col("zhvi_all_homes") - F.col("zhvi_1y_ago")) / F.col("zhvi_1y_ago")) * 100
        )
    )
    
    # Get latest date with YoY data
    latest_date = with_growth.agg(F.max("date")).collect()[0][0]
    
    return (with_growth
        .filter(F.col("date") == latest_date)
        .filter(F.col("region_level") == "city")
        .select(
            "region_name",
            "region_level",
            "zhvi_all_homes",
            "zhvi_1y_ago",
            "yoy_growth_pct",
            "date"
        )
        .orderBy(F.desc("yoy_growth_pct"))
        .limit(10)
        .withColumn("rank", F.row_number().over(Window.orderBy(F.desc("yoy_growth_pct"))))
    )

In [0]:
# ==================== AGGREGATION: YEARLY TREND ====================
@dlt.table(
    name="agg_yearly_trend",
    comment="Yearly trend summary - national/regional level metrics"
)
def agg_yearly_trend():
    fact = dlt.read("fact_housing_metrics")
    
    yearly = (fact
        .groupBy("year", "region_level")
        .agg(
            F.count("*").alias("total_observations"),
            F.countDistinct("region_name").alias("region_count"),
            F.avg("zhvi_all_homes").alias("avg_zhvi"),
            F.avg("zri_all_homes").alias("avg_zri"),
            F.avg("median_listing_price_all_homes").alias("avg_listing_price"),
            F.avg("price_to_rent_ratio_all_homes").alias("avg_price_to_rent"),
            F.avg("pct_of_homes_increasing_in_values_all_homes").alias("avg_pct_increasing"),
            F.avg("pct_of_homes_decreasing_in_values_all_homes").alias("avg_pct_decreasing")
        )
    )
    
    # Calculate YoY change
    window = Window.partitionBy("region_level").orderBy("year")
    
    return (yearly
        .withColumn("prev_avg_zhvi", F.lag("avg_zhvi").over(window))
        .withColumn("yoy_zhvi_change_pct",
            F.when(F.col("prev_avg_zhvi").isNotNull(),
                ((F.col("avg_zhvi") - F.col("prev_avg_zhvi")) / F.col("prev_avg_zhvi")) * 100
            )
        )
        .drop("prev_avg_zhvi")
        .orderBy("year", "region_level")
    )

In [0]:
# ==================== AGGREGATION: MARKET HEALTH SCORE ====================
@dlt.table(
    name="agg_market_health",
    comment="Market health indicators by region (latest snapshot)"
)
def agg_market_health():
    fact = dlt.read("fact_housing_metrics")
    
    # Get latest date
    latest_date = fact.agg(F.max("date")).collect()[0][0]
    
    return (fact
        .filter(F.col("date") == latest_date)
        .filter(F.col("region_level").isin("city", "metro"))
        .filter(F.col("zhvi_all_homes").isNotNull())
        .withColumn("affordability_score",
            # Lower price-to-rent = more affordable (normalize 0-100)
            F.when(F.col("price_to_rent_ratio_all_homes").isNotNull(),
                F.greatest(F.lit(0), F.lit(100) - F.col("price_to_rent_ratio_all_homes") * 3)
            ).otherwise(F.lit(50))
        )
        .withColumn("growth_momentum",
            # % of homes increasing in value
            F.coalesce(F.col("pct_of_homes_increasing_in_values_all_homes"), F.lit(50))
        )
        .withColumn("market_health_score",
            # Simple average of indicators
            (F.col("affordability_score") + F.col("growth_momentum")) / 2
        )
        .select(
            "region_name",
            "region_level",
            "zhvi_all_homes",
            "price_to_rent_ratio_all_homes",
            "affordability_score",
            "growth_momentum",
            "market_health_score",
            "date"
        )
        .orderBy(F.desc("market_health_score"))
    )